# GOV-01 E6: per-class threshold calibration

This notebook does **not** train a new model. It reuses the completed E6 model, predicts only on validation images, and chooses a separate confidence cutoff for each road condition. The protected test split is never loaded.

Before running it, make sure these two files are in `My Drive/GOV-01`:

- `v2_multilabel_n_rdd_expansion.zip`
- `v2_multilabel_e6_n_rdd_mobilenetv2_output.zip`


In [ ]:
# Connect Google Drive and check the two required files.
from pathlib import Path
import csv
import json
import shutil
import zipfile

from google.colab import drive
drive.mount('/content/drive')

DATASET_FOLDER = 'v2_multilabel_n_rdd_expansion'
CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover', 'unpaved_road', 'road_marking', 'speed_bump')
DRIVE_ROOT = Path('/content/drive/MyDrive/GOV-01')
DRIVE_DATASET_ZIP = DRIVE_ROOT / f'{DATASET_FOLDER}.zip'
DRIVE_E6_OUTPUT_ZIP = DRIVE_ROOT / 'v2_multilabel_e6_n_rdd_mobilenetv2_output.zip'
WORK_ROOT = Path('/content')
LOCAL_DATA_ROOT = WORK_ROOT / DATASET_FOLDER
LOCAL_MODEL_ROOT = WORK_ROOT / 'v2_multilabel_e6_calibration_model'
MODEL_FILENAME = 'e6_n_rdd_mobilenetv2_best.keras'
MODEL_PATH = LOCAL_MODEL_ROOT / MODEL_FILENAME
OUTPUT_DIRECTORY = WORK_ROOT / 'v2_multilabel_e6_threshold_calibration_output'

if not DRIVE_DATASET_ZIP.is_file():
    raise FileNotFoundError(f'Missing required dataset ZIP in Google Drive: {DRIVE_DATASET_ZIP}')
if DRIVE_E6_OUTPUT_ZIP.is_file():
    E6_OUTPUT_ZIP = DRIVE_E6_OUTPUT_ZIP
else:
    # This is only a 36 MB file. Use it if the earlier training output was downloaded but not copied to Drive.
    from google.colab import files
    print(f'{DRIVE_E6_OUTPUT_ZIP.name} is not in Drive. Choose the downloaded ZIP from this computer now.')
    uploaded = files.upload()
    if DRIVE_E6_OUTPUT_ZIP.name not in uploaded:
        raise FileNotFoundError(f'Upload exactly this file: {DRIVE_E6_OUTPUT_ZIP.name}')
    E6_OUTPUT_ZIP = WORK_ROOT / DRIVE_E6_OUTPUT_ZIP.name
    E6_OUTPUT_ZIP.write_bytes(uploaded[DRIVE_E6_OUTPUT_ZIP.name])

print('Dataset ZIP:', DRIVE_DATASET_ZIP)
print('Saved E6 output ZIP:', E6_OUTPUT_ZIP)
print('Protected test is deliberately not loaded by this notebook.')


In [ ]:
# Reuse a complete validation copy when it already exists; otherwise extract validation only.
def validation_is_complete(dataset_root):
    split_root = dataset_root / 'validation'
    labels_path = split_root / 'labels.csv'
    if not labels_path.is_file() or not (split_root / 'images').is_dir():
        return False
    with labels_path.open(newline='', encoding='utf-8') as source:
        rows = list(csv.DictReader(source))
    return bool(rows) and all((split_root / Path(row['materialized_image'].replace(chr(92), '/'))).is_file() for row in rows)

def safe_members(archive, members, destination_root):
    destination_root = destination_root.resolve()
    for member in members:
        destination = (destination_root / member.filename).resolve()
        if destination != destination_root and destination_root not in destination.parents:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')

if validation_is_complete(LOCAL_DATA_ROOT):
    print('Using the verified validation copy already in this Colab session.')
else:
    if LOCAL_DATA_ROOT.exists():
        print('Removing an incomplete temporary validation copy.')
        shutil.rmtree(LOCAL_DATA_ROOT)
    print('Extracting validation images only. Training and protected-test images are not extracted.')
    validation_prefix = f'{DATASET_FOLDER}/validation/'
    with zipfile.ZipFile(DRIVE_DATASET_ZIP) as archive:
        members = [member for member in archive.infolist() if member.filename.startswith(validation_prefix)]
        if not members:
            raise RuntimeError('The dataset ZIP has no validation folder at the expected location.')
        safe_members(archive, members, WORK_ROOT)
        archive.extractall(WORK_ROOT, members=members)
    if not validation_is_complete(LOCAL_DATA_ROOT):
        raise RuntimeError('Validation extraction is incomplete. Rerun this cell once after checking the Drive ZIP.')
    print('Validation extraction verified:', LOCAL_DATA_ROOT)

if not MODEL_PATH.is_file():
    LOCAL_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(E6_OUTPUT_ZIP) as archive:
        member = archive.getinfo(MODEL_FILENAME)
        safe_members(archive, [member], LOCAL_MODEL_ROOT)
        archive.extract(member, LOCAL_MODEL_ROOT)
if not MODEL_PATH.is_file():
    raise RuntimeError('The selected E6 model could not be extracted from the output ZIP.')

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print('E6 model ready:', MODEL_PATH)


In [ ]:
# Load validation labels and calculate E6 probabilities. A GPU is helpful but not required.
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score, precision_recall_fscore_support

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

def read_validation():
    split_root = LOCAL_DATA_ROOT / 'validation'
    paths, labels, masks = [], [], []
    with (split_root / 'labels.csv').open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            image_path = split_root / Path(row['materialized_image'].replace(chr(92), '/'))
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing validation image: {image_path}')
            label_row, mask_row = [], []
            for condition in CONDITIONS:
                known = int(row[f'{condition}_known'])
                value = row[f'{condition}_present']
                if known not in (0, 1) or (known == 0 and value != '') or (known == 1 and value not in ('0', '1')):
                    raise ValueError(f'Invalid masked label: {row["record_id"]} / {condition}')
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path))
            labels.append(label_row)
            masks.append(mask_row)
    return np.asarray(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32')

validation_paths, validation_labels, validation_masks = read_validation()
for index, condition in enumerate(CONDITIONS):
    print(f'{condition:16} known={int(validation_masks[:, index].sum()):5} positive={int((validation_labels[:, index] * validation_masks[:, index]).sum()):5}')

def load_image(path):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (224, 224))
    return tf.cast(image, tf.float32)

validation_images = tf.data.Dataset.from_tensor_slices(validation_paths)
validation_images = validation_images.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)
best_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
validation_probabilities = best_model.predict(validation_images, verbose=1)
print('Prediction shape:', validation_probabilities.shape)


In [ ]:
# Test confidence cutoffs from 0.05 to 0.95 and select the cutoff with the best validation F1 for each class.
import matplotlib.pyplot as plt

THRESHOLD_GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)
DEFAULT_THRESHOLD = 0.50
default_index = int(np.where(np.isclose(THRESHOLD_GRID, DEFAULT_THRESHOLD))[0][0])
calibration = {}
figure, axes = plt.subplots(2, 4, figsize=(16, 8))

for index, condition in enumerate(CONDITIONS):
    known = validation_masks[:, index].astype(bool)
    actual = validation_labels[known, index].astype(int)
    probabilities = validation_probabilities[known, index]
    scores = np.asarray([f1_score(actual, probabilities >= threshold, zero_division=0) for threshold in THRESHOLD_GRID])
    best_score = scores.max()
    tied = np.flatnonzero(np.isclose(scores, best_score))
    best_index = min(tied, key=lambda item: (abs(float(THRESHOLD_GRID[item]) - DEFAULT_THRESHOLD), float(THRESHOLD_GRID[item])))
    default_precision, default_recall, default_f1, _ = precision_recall_fscore_support(actual, probabilities >= DEFAULT_THRESHOLD, average='binary', zero_division=0)
    selected_threshold = float(THRESHOLD_GRID[best_index])
    selected_precision, selected_recall, selected_f1, _ = precision_recall_fscore_support(actual, probabilities >= selected_threshold, average='binary', zero_division=0)
    calibration[condition] = {
        'known_images': int(len(actual)), 'positive_images': int(actual.sum()),
        'default_threshold': DEFAULT_THRESHOLD, 'default_precision': float(default_precision),
        'default_recall': float(default_recall), 'default_f1': float(default_f1),
        'selected_threshold': selected_threshold, 'selected_precision': float(selected_precision),
        'selected_recall': float(selected_recall), 'selected_f1': float(selected_f1),
        'f1_change': float(selected_f1 - default_f1)
    }
    axis = axes.flat[index]
    axis.plot(THRESHOLD_GRID, scores, linewidth=1.5)
    axis.axvline(DEFAULT_THRESHOLD, color='gray', linestyle='--', label='default 0.50')
    axis.axvline(selected_threshold, color='green', linestyle='--', label=f'selected {selected_threshold:.2f}')
    axis.set_title(condition)
    axis.set_xlabel('confidence cutoff')
    axis.set_ylabel('validation F1')
    axis.legend(fontsize=8)

axes.flat[-1].axis('off')
figure.tight_layout()
figure.savefig(OUTPUT_DIRECTORY / 'e6_threshold_calibration_curves.png', dpi=150)

default_macro_f1 = float(np.mean([item['default_f1'] for item in calibration.values()]))
selected_macro_f1 = float(np.mean([item['selected_f1'] for item in calibration.values()]))
result = {
    'experiment': 'v2_multilabel_e6_threshold_calibration',
    'selection_data': 'validation only',
    'protected_test_loaded': False,
    'default_macro_f1': default_macro_f1,
    'calibrated_macro_f1': selected_macro_f1,
    'macro_f1_change': selected_macro_f1 - default_macro_f1,
    'per_condition': calibration
}
(OUTPUT_DIRECTORY / 'e6_threshold_calibration.json').write_text(json.dumps(result, indent=2) + '\n')
with (OUTPUT_DIRECTORY / 'e6_threshold_calibration.csv').open('w', newline='', encoding='utf-8') as destination:
    writer = csv.DictWriter(destination, fieldnames=['condition', *next(iter(calibration.values())).keys()])
    writer.writeheader()
    for condition, values in calibration.items():
        writer.writerow({'condition': condition, **values})

for condition, values in calibration.items():
    print(f'{condition:16} cutoff={values["selected_threshold"]:.2f}  F1: {values["default_f1"]:.3f} -> {values["selected_f1"]:.3f}  change={values["f1_change"]:+.3f}')
print(f'\nMacro F1: {default_macro_f1:.4f} -> {selected_macro_f1:.4f}  change={selected_macro_f1 - default_macro_f1:+.4f}')
print('Protected test remains untouched.')


In [ ]:
# Save the small calibration report to Drive and automatically download it to this computer.
from google.colab import files

result_zip = WORK_ROOT / 'v2_multilabel_e6_threshold_calibration_output.zip'
if result_zip.exists():
    result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_DIRECTORY)
drive_output = DRIVE_ROOT / result_zip.name
shutil.copy2(result_zip, drive_output)
print('Saved a backup in Google Drive:', drive_output)
print('Starting browser download:', result_zip.name)
files.download(str(result_zip))
